Won't be deployed until I can move away from Render (also why its a jupyter notebook)

Check if valid:
    - Is screenshot of TEC
    - Is on results screen

Check for:
    - Both sides: APM, DPM, SR, Score
    - One side: Time

In [1]:
# imports

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import shutil
import glob
from sklearn.model_selection import train_test_split

2025-11-07 20:16:21.172509: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-07 20:16:21.253602: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-07 20:16:21.253698: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-07 20:16:21.257547: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-07 20:16:21.282687: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# constants
INP_DIR:str = "data/validation"
DATA_DIR:str = "data/validation_processed/training"
TEST_DIR:str = "data/validation_processed/testing"
TEMP_DIR:str = "data/validation_processed/temp"
SEED:int = 136
TRAIN_TEST_SPLIT:float = 0.2

# left top right bottom
YELLOW_FIELD:tuple[float,float,float,float] = (0.0, 0.0, 0.6, 1.0)
BLUE_FIELD:tuple[float,float,float,float] = (0.5, 0.0, 1.0, 1.0)

In [3]:
# create directories

if not os.path.exists(TEST_DIR):
    os.makedirs(TEST_DIR)

if not os.path.exists(f"{TEST_DIR}/0_invalid"):
    os.makedirs(f"{TEST_DIR}/0_invalid")

if not os.path.exists(f"{TEST_DIR}/1_tec_results"):
    os.makedirs(f"{TEST_DIR}/1_tec_results")


if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

if not os.path.exists(f"{DATA_DIR}/0_invalid"):
    os.makedirs(f"{DATA_DIR}/0_invalid")

if not os.path.exists(f"{DATA_DIR}/1_tec_results"):
    os.makedirs(f"{DATA_DIR}/1_tec_results")


if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

if not os.path.exists(f"{TEMP_DIR}/0_invalid"):
    os.makedirs(f"{TEMP_DIR}/0_invalid")

if not os.path.exists(f"{TEMP_DIR}/1_tec_results"):
    os.makedirs(f"{TEMP_DIR}/1_tec_results")

In [4]:
# data modification

def crop_image(path_in:str, class_dir:str, name:str) -> None:
    img = Image.open(path_in)

    width, height = img.size

    image_size = (width, height, width, height)

    yellow_image = (width*YELLOW_FIELD[0],
                    0,
                    width*YELLOW_FIELD[2],
                    height
                    )
    
    blue_image = (width*BLUE_FIELD[0],
                    0,
                    width*BLUE_FIELD[2],
                    height
                    )

    yellow = img.crop(yellow_image)
    yellow.save(f"{TEMP_DIR}/{class_dir}/YELLOW_{name}")

    blue = img.crop(blue_image)
    blue.save(f"{TEMP_DIR}/{class_dir}/BLUE_{name}")

for class_dir in os.listdir(INP_DIR):
    # goes through each folder

    path:str = f"{INP_DIR}/{class_dir}"
    for file in os.listdir(path):
        crop_image(f"{path}/{file}", class_dir, file)

In [5]:
# now we need to split the data
files = []
labels = []


# first, index everything 
for class_dir in os.listdir(TEMP_DIR):
    class_path = f"{TEMP_DIR}/{class_dir}"

    # makes a list of all the files in the path
    img_files = glob.glob(f"{class_path}/*")

    label:int = 1 if class_dir.startswith("1") else 0

    files.extend(img_files)
    labels.extend([label]*len(img_files))

# split them with train test split

train_val_files, test_files, y, y = train_test_split(
    files,
    labels,
    test_size=TRAIN_TEST_SPLIT,
    random_state=SEED,
    stratify=labels
)

for file_path in train_val_files:
    new_path = file_path.replace("/temp/", "/training/")
    shutil.move(file_path, new_path)

for file_path in test_files:
    new_path = file_path.replace("/temp/", "/testing/")
    shutil.move(file_path, new_path)

In [ ]:
# load the data

IMG_SIZE:tuple[int, int] = (224, 224)
BATCH_SIZE:int = 16 # my 6GB of RAM needs limits haha

print("Loading and resizing to", IMG_SIZE)

train_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


valid_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_tec_results.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_tec_results = train_tec_results.cache().prefetch(buffer_size=AUTOTUNE)
valid_tec_results = valid_tec_results.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\n Found classes: {class_names}")

In [ ]:
# Construct a Mobile Net 
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape= IMG_SIZE + (3,),
    alpha=1.0, # determines how much to cut down the model, 1 is fully intact
    minimalistic=False,
    include_top=False, # removes classification layer so we can add our own
    weights="imagenet",
    input_tensor=None,
    classes=1000,
    pooling=None,
    dropout_rate=0.2,
    classifier_activation="softmax",
    include_preprocessing=True,
)

# binary classification outputs
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediciton_layer = tf.keras.layers.Dense(1, activation="sigmoid")

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

processed_input = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
base_model_out = base_model(processed_input, training=False)
pooled_out = global_average_layer(base_model_out)
dropped_out = tf.keras.layers.Dropout(0.2)(pooled_out)

outputs = prediciton_layer(dropped_out)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

In [20]:
# load the data but different sizing 


IMG_SIZE:tuple[int, int] = (380, 380)
BATCH_SIZE:int = 1 # my 6GB of RAM needs limits haha


print("Loading and resizing to", IMG_SIZE)

train_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


valid_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_tec_results.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_tec_results = train_tec_results.cache().prefetch(buffer_size=AUTOTUNE)
valid_tec_results = valid_tec_results.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\n Found classes: {class_names}")

Loading and resizing to (380, 380)
Found 251 files belonging to 2 classes.
Using 201 files for training.
Found 251 files belonging to 2 classes.
Using 50 files for validation.

 Found classes: ['0_invalid', '1_tec_results']


In [21]:
# Construct a EfficientNetB4
base_model = tf.keras.applications.EfficientNetB4(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=IMG_SIZE + (3,),
    pooling=None,
    classes=1000,
    classifier_activation="softmax",
)

# binary classification outputs
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediciton_layer = tf.keras.layers.Dense(1, activation="sigmoid")

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

processed_input = tf.keras.applications.efficientnet.preprocess_input(inputs)
base_model_out = base_model(processed_input, training=False)
pooled_out = global_average_layer(base_model_out)
dropped_out = tf.keras.layers.Dropout(0.2)(pooled_out)

outputs = prediciton_layer(dropped_out)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

2025-11-07 20:25:49.978965: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


71686520/71686520 [==============================] - 17s 0us/step
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 380, 380, 3)]     0         
                                                                 
 efficientnetb4 (Functional  (None, 12, 12, 1792)      17673823  
 )                                                               
                                                                 
 global_average_pooling2d_3  (None, 1792)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dropout_3 (Dropout)         (None, 1792)              0         
                                                                 
 dense_3 (Dense)             (None, 1)                 1793      
                                                           

In [22]:
print("It's training time!")

checkpointer = tf.keras.callbacks.ModelCheckpoint(
  filepath='tec_check_weights.keras',
  monitor='accuracy',
  verbose=1,
  save_best_only=True
)

It's training time!


In [23]:
model.fit(
    train_tec_results,
    epochs=10,
    validation_data=valid_tec_results,
    callbacks=checkpointer,
    verbose=2
)

Epoch 1/10


2025-11-07 20:27:09.979957: I external/local_xla/xla/service/service.cc:168] XLA service 0x7961a40a95c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-07 20:27:09.980017: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 SUPER, Compute Capability 7.5
2025-11-07 20:27:09.986132: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1762568830.087206   93552 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: accuracy improved from -inf to 0.83085, saving model to tec_check_weights.keras
201/201 - 103s - loss: 0.3732 - accuracy: 0.8308 - val_loss: 0.3799 - val_accuracy: 0.8600 - 103s/epoch - 510ms/step
Epoch 2/10

Epoch 2: accuracy improved from 0.83085 to 0.92537, saving model to tec_check_weights.keras
201/201 - 31s - loss: 0.1866 - accuracy: 0.9254 - val_loss: 0.2496 - val_accuracy: 0.9200 - 31s/epoch - 152ms/step
Epoch 3/10

Epoch 3: accuracy improved from 0.92537 to 0.95025, saving model to tec_check_weights.keras
201/201 - 30s - loss: 0.1359 - accuracy: 0.9502 - val_loss: 0.0217 - val_accuracy: 1.0000 - 30s/epoch - 150ms/step
Epoch 4/10

Epoch 4: accuracy did not improve from 0.95025
201/201 - 27s - loss: 0.1687 - accuracy: 0.9403 - val_loss: 0.0927 - val_accuracy: 0.9600 - 27s/epoch - 133ms/step
Epoch 5/10

Epoch 5: accuracy did not improve from 0.95025
201/201 - 27s - loss: 0.2147 - accuracy: 0.9403 - val_loss: 0.0688 - val_accuracy: 1.0000 - 27s/epoch - 135ms/step
Epoch 6

In [24]:
# load testing

test_set = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_set = test_set.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

loss, acc = model.evaluate(test_set, verbose=2)

print(f"\nFinal Test Accuracy: {acc * 100:.2f}%")

Found 63 files belonging to 2 classes.
63/63 - 2s - loss: 0.0027 - accuracy: 1.0000 - 2s/epoch - 31ms/step

Final Test Accuracy: 100.00%
